# PaySim Preprocessing & Feature Engineering
Tuân thủ quy tắc chống rò rỉ dữ liệu (No-Balance Leakage Rule):
- Chỉ lọc 2 loại giao dịch có gian lận: `TRANSFER` và `CASH_OUT`.
- Loại bỏ các cột số dư `oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest` (theo lưu ý từ tác giả Kaggle để tránh data leakage).
- Tạo 4 nhóm đặc trưng:
  1. Thời gian: `hourOfDay`, `dayOfWeek`, `isNight`
  2. Số tiền: `logAmount`, `isCappedAmount`, `isOverCap`, `isZeroAmount`
  3. Hành vi tích lũy: `destTxCount`, `origTxCount`, `isFirstTimeDest`, `amountToDestMeanRatio`
  4. Lưu lượng theo giờ & Twin transaction: `stepTxCount`, `sameStepAmountCount`, `isPairedCashOut`


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def get_paysim_path():
    candidates = [
        '../../data/paysim/PS_20174392719_1491204439457_log.csv',
        '../data/paysim/PS_20174392719_1491204439457_log.csv',
        'data/paysim/PS_20174392719_1491204439457_log.csv',
        'E:/HUST/CLB/GDG/data/paysim/PS_20174392719_1491204439457_log.csv',
        'E:/HUST/CLB/GDG/Nhom01_GDG-main/data/paysim/PS_20174392719_1491204439457_log.csv'
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError('Khong tim thay file PS_20174392719_1491204439457_log.csv')


In [ ]:
def cum_count(df, group_cols, name):
    df[name] = (df.groupby(group_cols).cumcount() + 1).astype("int32")
    return df

def feature_engineering_paysim(df_in):
    df = df_in.copy()
    
    # 1. Time features
    df["hourOfDay"] = (df["step"] % 24).astype("int8")
    df["isNight"] = ((df["hourOfDay"] >= 0) & (df["hourOfDay"] <= 6)).astype("int8")
    df["dayOfWeek"] = ((df["step"] // 24) % 7).astype("int8")
    
    # 2. Amount features
    df["logAmount"] = np.log1p(df["amount"]).astype("float32")
    df["isCappedAmount"] = (df["amount"] == 10_000_000.0).astype("int8")
    df["isOverCap"] = (df["amount"] > 10_000_000.0).astype("int8")
    df["isZeroAmount"] = (df["amount"] == 0.0).astype("int8")
    
    # Sort by step to avoid future leakage
    df = df.sort_values("step", kind="mergesort").reset_index(drop=True)
    
    # 3. Account cumulative behavior
    df = cum_count(df, "nameDest", "destTxCount")
    df = cum_count(df, "nameOrig", "origTxCount")
    df["isFirstTimeDest"] = (df["destTxCount"] == 1).astype("int8")
    dest_cum_sum = df.groupby("nameDest")["amount"].cumsum()
    df["destMeanAmount"] = dest_cum_sum / df["destTxCount"]
    df["amountToDestMeanRatio"] = (df["amount"] / (df["destMeanAmount"] + 1e-5)).astype("float32")
    
    # 4. Step density & twin transaction features
    df = cum_count(df, "step", "stepTxCount")
    df = cum_count(df, ["step", "amount"], "sameStepAmountCount")
    df["isPairedCashOut"] = ((df["sameStepAmountCount"] == 2) & (df["type"] == 1)).astype("int8")
    
    return df


In [ ]:
def get_data(sample_size=300000, test_size=0.2, random_state=42):
    path = get_paysim_path()
    df = pd.read_csv(path)
    # Loc 2 loai co gian lan: TRANSFER va CASH_OUT
    df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
    
    # Loai bo 4 cot so du (chong leakage theo Kaggle rules)
    df = df.drop(columns=['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'])
    df['type'] = df['type'].map({'TRANSFER': 0, 'CASH_OUT': 1})
    
    if sample_size and sample_size < len(df):
        df = df.sample(n=sample_size, random_state=random_state)
        
    df_fe = feature_engineering_paysim(df)
    
    cols_to_drop = ['step', 'amount', 'nameOrig', 'nameDest', 'destMeanAmount', 'isFraud']
    X = df_fe.drop(columns=[c for c in cols_to_drop if c in df_fe.columns])
    y = df_fe['isFraud']
    
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)

if __name__ == '__main__':
    X_train, X_test, y_train, y_test = get_data(sample_size=50000)
    print('Train shape:', X_train.shape, 'Fraud rate:', y_train.mean())
    print('Test shape: ', X_test.shape, 'Fraud rate: ', y_test.mean())
